In [0]:
# 1 - Criação do Database da camada Silver:

"""
Descrição geral: Import de todos os módulos necessários para as transformações desta parte do notebook de uma vez, no topo, garantindo que não aconteçam erros de importação no meio da execução e facilitando a leitura.

Novamente, usamos IF NOT EXISTS para garantir a idempotência, ou seja, evitar problemas caso o Schema já exista.

pyspark.sql.functions (F):
- Funções nativas distribuídas do Spark (col, when, lit, datediff, upper, coalesce, current_timestamp, try_to_timestamp, etc.).
- Rodam nos workers do cluster — muito mais eficientes que UDFs Python.
  
pyspark.sql.window.Window:
- Define janelas de particionamento e ordenação para Window Functions.
- Usado na deduplicação sênior de dim_consumidores.
  
pyspark.sql.types:
- Tipos explícitos para cast() — StringType, IntegerType, DoubleType, TimestampType, DateType.
"""

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (StringType, IntegerType, DoubleType, TimestampType, DateType)

spark.sql("""
    CREATE DATABASE IF NOT EXISTS silver
    COMMENT 'Camada Silver — Dados limpos, tipados e com regras de negócio aplicadas'
""")

print("Imports concluídos.")
print("Banco de dados 'silver' criado")
print()
print("Databases disponíveis:")
spark.sql("SHOW DATABASES").show(truncate=False)

In [0]:
# 2 - Tratamento da tabela de consumidores:

"""
Descrição geral do passo a passo que está acontecendo na célula:

Passo 1: Leitura da bronze:
- spark.table('bronze.tb_customers') lê a tabela Delta registrada no metastore. Nenhuma modificação é feita na Bronze.
  
Passo 2: Renomeação e tipagem (.select com alias e cast):
Usamos .select() em vez de múltiplos .withColumnRenamed() porque:
- É mais legível quando há muitas colunas para renomear.
- Permite aplicar cast() inline, junto com o alias.
- Seleciona apenas as colunas necessárias e descarta o restante.
CEP como StringType:
- Na Bronze, inferSchema pode ter inferido o CEP como IntegerType.
- CEP '01310' viraria 1310 (inteiro) — perdendo o zero à esquerda.
- cast(StringType()) garante que o CEP seja sempre tratado como texto.
  
Passo 3: 
Upper Case (F.upper):
- Padroniza cidade e estado em maiúsculas.
- Evita duplicatas semânticas como 'São Paulo' vs 'SAO PAULO'.
- F.upper() é uma função nativa do Spark, ou seja, roda no cluster.
  
Passo 4 - Deduplicação Sênior com Window Function:
  
Window.partitionBy('id_consumidor'):
- Cria uma "janela" separada para cada id_consumidor distinto.
- Dentro de cada janela, os registros são ordenados. 

.orderBy(F.col('timestamp_ingestion').desc()):
- Ordena do mais recente para o mais antigo dentro de cada janela.
- O registro mais recente fica na posição 1.

F.row_number().over(janela_dedup):
- Atribui o número 1 ao registro mais recente de cada consumidor.
- Registros duplicados recebem números 2, 3, 4...
  
Passo 5: Filtro e Drop da coluna auxiliar:
- Mantém apenas o registro com rn=1 (o mais recente por consumidor).
- .drop('rn'): remove a coluna auxiliar, já que não faz parte do modelo.
"""

# Definição da janela de deduplicação: 

# A partição ocorre por id_consumidor e a ordenação ocorre pelo timestamp mais recente
janela_dedup_consumidor = (
    Window
        .partitionBy("id_consumidor")
        .orderBy(F.col("timestamp_ingestion").desc())
)

# Leitura, transformação e gravação:
df_consumidores = (
    spark.table("bronze.tb_customers")

    # Renomeação de colunas para português com tipagem explícita:
    .select(
        F.col("customer_id").cast(StringType()).alias("id_consumidor"),
        F.col("customer_zip_code_prefix").cast(StringType()).alias("prefixo_cep"),
        F.col("customer_city").cast(StringType()).alias("cidade"),
        F.col("customer_state").cast(StringType()).alias("estado"),
        F.col("customer_name").cast(StringType()).alias("nome_consumidor"),
        F.col("timestamp_ingestion"),
    )

    # Upper Case em cidade e estado:
    .withColumn("cidade", F.upper(F.col("cidade")))
    .withColumn("estado", F.upper(F.col("estado")))

    # Deduplicação Sênior: row_number dentro de cada partição de id_consumidor
    .withColumn("rn", F.row_number().over(janela_dedup_consumidor))
    .filter(F.col("rn") == 1)
    .drop("rn")  # Coluna auxiliar — removida após o filtro
)

# Gravação na Silver
(
    df_consumidores.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("silver.dim_consumidores")
)

total = spark.table("silver.dim_consumidores").count()
print(f"silver.dim_consumidores criada: {total:,} registros únicos")
print()
print("Amostra:")
display(spark.table("silver.dim_consumidores").limit(5))